In [1]:
import os, glob,shutil,json
from pathlib import Path
KOK = Path(r"gol")
HEDEF = KOK / "dagitik"
HEDEF.mkdir(parents=True, exist_ok=True)
adaylar = glob.glob(str(KOK / "yil=*" / "*.parquet"))
if not adaylar:
    raise FileNotFoundError(f"Parquet bulunamadi: {KOK}")
KAYNAK = max(adaylar, key=os.path.getsize)
BOYUT = os.path.getsize(KAYNAK)
print("Kaynak :", KAYNAK)
print("Hedef  :", HEDEF.resolve())
print(f"Boyut  : {BOYUT / 1024**2:.2f} MB")


Kaynak : gol\yil=2023\129f0eb6e7bf42bd94c01729578417d2-0.parquet
Hedef  : C:\Users\yusuf\Desktop\bigdata\gol\dagitik
Boyut  : 5.82 MB


In [2]:
PARCA_BOYUTU = 512 * 1024 
parcalar = []
with open(KAYNAK, "rb") as f:
    i = 0
    while True:
        veri = f.read(PARCA_BOYUTU)
        if not veri:
            break
        parcalar.append((f"parca_{i:04d}", veri))
        i += 1
print(f"Parca sayisi : {len(parcalar)}")
print(f"Son parca    : {len(parcalar[-1][1]) / 1024**2:.2f} MB")


Parca sayisi : 12
Son parca    : 0.32 MB


In [3]:
import pyarrow.parquet as pq

dosyalar = glob.glob(str(KOK / "yil=*" / "*.parquet"))
print("Dosya sayısı :", len(dosyalar))
print("Toplam boyut :", sum(os.path.getsize(d) for d in dosyalar) / 1024**2, "MB")
print("Satır sayısı :", pq.ParquetFile(KAYNAK).metadata.num_rows)

Dosya sayısı : 719
Toplam boyut : 1045.717851638794 MB
Satır sayısı : 40328


In [4]:
MAKINELER = [HEDEF / "makine1", HEDEF / "makine2", HEDEF / "makine3"]
KOPYA = 2
for m in MAKINELER:
    shutil.rmtree(m, ignore_errors=True)
    m.mkdir(parents=True)
katalog = {}
for sira, (ad, veri) in enumerate(parcalar):
    yerler = []
    for k in range(KOPYA):
        hedef = MAKINELER[(sira + k) % len(MAKINELER)]
        (hedef / ad).write_bytes(veri)
        yerler.append(hedef.name)
    katalog[ad] = {"yerler": yerler, "boyut": len(veri)}
with open(HEDEF / "katalog.json", "w") as f:
    json.dump({"kaynak": KAYNAK, "parca_boyutu": PARCA_BOYUTU,
               "kopya": KOPYA, "parcalar": katalog}, f, indent=2)
print("Bölme işlemi tamamlandı")


Bölme işlemi tamamlandı


In [6]:
import os, glob,shutil,json
from pathlib import Path
KOK = Path(r"gol")
HEDEF = KOK / "dagitik"
HEDEF.mkdir(parents=True, exist_ok=True)
adaylar = glob.glob(str(KOK / "yil=*" / "*.parquet"))
if not adaylar:
    raise FileNotFoundError(f"Parquet bulunamadi: {KOK}")
KAYNAK = max(adaylar, key=os.path.getsize)
BOYUT = os.path.getsize(KAYNAK)
PARCA_BOYUTU = 512 * 1024 
parcalar = []
with open(KAYNAK, "rb") as f:
    i = 0
    while True:
        veri = f.read(PARCA_BOYUTU)
        if not veri:
            break
        parcalar.append((f"parca_{i:04d}", veri))
        i += 1
MAKINELER = [HEDEF / "makine1", HEDEF / "makine2", HEDEF / "makine3"]
for m in MAKINELER:
    dosyalar = sorted(os.listdir(m))
    numaralar = [int(x.split("_")[1]) for x in dosyalar]
    yer = sum(os.path.getsize(m / x) for x in dosyalar)
    print(f"{m.name}: {numaralar}  ({yer / 1024**2:.1f} MB)")
toplam = sum(sum(os.path.getsize(m / x) for x in os.listdir(m)) for m in MAKINELER)
print(f"Diskte toplam : {toplam / 1024**2:.1f} MB")
print(f"Kac kat yer   : {toplam / BOYUT:.2f}")


makine1: [0, 2, 3, 5, 6, 8, 9, 11]  (3.8 MB)
makine2: [0, 1, 3, 4, 6, 7, 9, 10]  (4.0 MB)
makine3: [1, 2, 4, 5, 7, 8, 10, 11]  (3.8 MB)
Diskte toplam : 11.6 MB
Kac kat yer   : 2.00


In [1]:
import os, json, hashlib, shutil
from pathlib import Path

BASE = Path(r"C:\Users\yusuf\Desktop\bigdata")
HEDEF = BASE / "gol" / "dagitik"
os.chdir(HEDEF)

if not os.path.exists("katalog.json"):
    raise FileNotFoundError("katalog.json yok. Once Adim 2'yi calistirin.")

with open("katalog.json") as f:
    KAYNAK = str(BASE / json.load(f)["kaynak"])
print("Kaynak :", KAYNAK)

# buradan sonrası (birlestir, parmak_izi, asil = ..., input ...) aynen kalsın
def birlestir(hedef_dosya="geri_gelen.parquet"):
    with open("katalog.json") as f:
        kat = json.load(f)
    kayip = []
    with open(hedef_dosya, "wb") as cikti:
        for ad, bilgi in kat["parcalar"].items():
            bulundu = None
            for yer in bilgi["yerler"]:
                yol = f"{yer}/{ad}"
                if os.path.exists(yol):
                    bulundu = yol
                    break
            if bulundu is None:
                kayip.append(ad)
                continue
            with open(bulundu, "rb") as f:
                cikti.write(f.read())
    return kayip
def parmak_izi(yol):
    h = hashlib.sha256()
    with open(yol, "rb") as f:
        for p in iter(lambda: f.read(1024 * 1024), b""):
            h.update(p)
    return h.hexdigest()
# Hiçbir şey silmeden
asil = parmak_izi(KAYNAK)
kayip = birlestir()
print("Kayip parca :", len(kayip))
print("Ayni mi     :", parmak_izi("geri_gelen.parquet") == asil)
# makine2 bozuldu
input("makine2 silinecek. Klasore bakin, sonra Enter'a basin...")
shutil.rmtree("makine2", ignore_errors=True)
kayip = birlestir()
print("Kayip parca :", len(kayip))
print("Ayni mi     :", parmak_izi("geri_gelen.parquet") == asil)


Kaynak : C:\Users\yusuf\Desktop\bigdata\gol\yil=2023\129f0eb6e7bf42bd94c01729578417d2-0.parquet
Kayip parca : 0
Ayni mi     : True
Kayip parca : 0
Ayni mi     : True
